# Tetris OpenEnv — Per-Piece GRPO Training

Train an LLM to play Tetris: model sees the board before **every piece**, outputs actions until piece locks, learns via GRPO-style policy gradient.

**Environment**: Local Tetris engine (game_engine.py from OpenEnv)
**Model**: Qwen2.5-3B-Instruct + LoRA (r=16)
**Training**: Custom REINFORCE/GRPO loop — 8 games/iteration, 100 iterations
**Runtime**: L4 GPU (Colab)

In [ ]:
# Cell 1: Install dependencies
!pip install peft accelerate -q
!pip install -U bitsandbytes -q
!python -c "import bitsandbytes; print(bitsandbytes.__version__)"


In [ ]:
# Cell 2: Load Qwen2.5-3B-Instruct + LoRA (INT4 / QLoRA)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for batched generation

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Cell 3: Game engine + constants + prompt builder
import random
import torch.nn.functional as F

# Download game engine from repo
!wget -q -O game_engine.py "https://raw.githubusercontent.com/OutOfMystic/tetris-openenv/main/src/tetris_env/server/game_engine.py?$(date +%s)"
from game_engine import TetrisEnv, __version__
print(f"Game engine v{__version__}")

# === Constants ===
MAX_ACTIONS_PER_PIECE = 20   # max tokens model can output per piece
MAX_PIECES_PER_GAME = 200    # max pieces (model calls) per game
GAMES_PER_ITER = 8           # games per training iteration (same seed)
NUM_ITERATIONS = 100         # total training iterations
TEMPERATURE = 0.7

# Training reward modifiers (on top of engine per-step rewards)
LR_PENALTY = -0.1            # per L/R move
PIECE_PLACED_BONUS = 2.0     # per piece successfully placed
NO_PLACE_PENALTY = -10.0     # if 20 tokens exhausted without placing
LINE_CLEAR_BONUS = 100.0     # per line cleared (+100/+200/+300/+400)

# Action mapping
ACTION_CHARS = ['L', 'R', 'C', 'W', 'D', 'S']
ACTION_TO_ENGINE = {
    'L': 'left', 'R': 'right', 'C': 'rotate_cw',
    'W': 'rotate_ccw', 'D': 'drop', 'S': 'down'
}

# Pre-compute token IDs for action characters
ACTION_TOKEN_IDS = []
for ch in ACTION_CHARS:
    ids = tokenizer.encode(ch, add_special_tokens=False)
    ACTION_TOKEN_IDS.append(ids[0])
    print(f"  '{ch}' -> token_id {ids[0]}")
ACTION_TOKEN_IDS = torch.tensor(ACTION_TOKEN_IDS, device=model.device)

# Token ID -> action char lookup
TOKEN_TO_CHAR = {ACTION_TOKEN_IDS[i].item(): ACTION_CHARS[i] for i in range(6)}

SYSTEM_PROMPT = """You play Tetris. You see a 10-wide board and a falling piece marked @.
Your job: move/rotate the piece, then drop it to fill rows without creating holes.

Actions (one letter per step):
  L = shift piece 1 cell left
  R = shift piece 1 cell right
  C = rotate piece 90° clockwise
  W = rotate piece 90° counter-clockwise
  S = move piece down 1 row (soft drop)
  D = hard drop — piece falls straight down to the lowest position and locks immediately

After every action except D, gravity pulls the piece down 1 row. If it can't move down, it locks.

Strategy: use L/R/C/W to position the piece, then D to lock it. Avoid leaving holes (empty cells under filled ones). Fill complete rows to clear lines."""

def build_prompt(result):
    return f"""Board (# = locked blocks, @ = your current piece):
{result['board']}

Current piece: {result['current_piece']}  Next piece: {result['next_piece']}
Score: {result['score']}  Lines cleared: {result['total_lines']}  Holes: {result['holes']}

Output your actions (e.g. L L C D):"""

print("\nGame engine loaded. Action tokens mapped.")


In [ ]:
# Cell 4: Core training functions — batched play + train
import time as _time

def play_games_batched(model, tokenizer, seed, num_games=GAMES_PER_ITER, temperature=TEMPERATURE):
    """
    Play num_games Tetris games IN PARALLEL (batched forward passes).
    All games use the same seed but diverge due to stochastic sampling.
    Returns list of game results for GRPO training.
    """
    # Initialize all environments
    envs = []
    for _ in range(num_games):
        env = TetrisEnv(seed=seed)
        env.reset(seed=seed)
        rng = random.Random(seed)
        moves = rng.randint(0, 4)
        direction = rng.choice(["left", "right"])
        for _ in range(moves):
            if env.done:
                break
            env.step(direction)
        envs.append(env)

    # Per-game tracking
    total_rewards = [0.0] * num_games
    total_steps = [0] * num_games
    pieces_counts = [0] * num_games
    pieces_data = [[] for _ in range(num_games)]
    game_active = [True] * num_games

    pad_id = tokenizer.pad_token_id

    while any(game_active):
        # Which games need a new piece?
        active_indices = [i for i in range(num_games)
                         if game_active[i] and not envs[i].done
                         and pieces_counts[i] < MAX_PIECES_PER_GAME]

        if not active_indices:
            break

        # Mark inactive games
        for i in range(num_games):
            if i not in active_indices:
                game_active[i] = False

        batch_size = len(active_indices)

        # Save current piece names for lock detection
        current_piece_names = [envs[i].current_piece_name for i in active_indices]
        lines_before = [envs[i].total_lines for i in active_indices]

        # Build prompts for all active games
        prompt_ids_list = []
        for i in active_indices:
            pieces_counts[i] += 1
            result = envs[i]._make_result(0)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_prompt(result)},
            ]
            ids = tokenizer.apply_chat_template(
                messages, return_tensors="pt", add_generation_prompt=True
            )
            if hasattr(ids, 'input_ids'):
                ids = ids['input_ids']
            prompt_ids_list.append(ids.squeeze(0))

        # Left-pad to same length
        max_len = max(p.shape[0] for p in prompt_ids_list)
        padded_input = torch.full((batch_size, max_len), pad_id, dtype=torch.long, device=model.device)
        attention_mask = torch.zeros(batch_size, max_len, dtype=torch.long, device=model.device)

        for j, p in enumerate(prompt_ids_list):
            seq_len = p.shape[0]
            padded_input[j, max_len - seq_len:] = p.to(model.device)
            attention_mask[j, max_len - seq_len:] = 1

        # Batch forward pass for prompt encoding
        with torch.no_grad():
            out = model(padded_input, attention_mask=attention_mask, use_cache=True)
            past_kv = out.past_key_values

        # Per-game autoregressive state
        action_ids_lists = [[] for _ in range(batch_size)]
        piece_placed = [False] * batch_size
        piece_done = [False] * batch_size

        # Autoregressive generation loop
        with torch.no_grad():
            for step in range(MAX_ACTIONS_PER_PIECE):
                logits = out.logits[:, -1, :]

                # Mask to 6 action tokens
                masked = torch.full_like(logits, float('-inf'))
                masked[:, ACTION_TOKEN_IDS] = logits[:, ACTION_TOKEN_IDS]
                probs = F.softmax(masked / temperature, dim=-1)

                # Sample actions for all games
                token_ids = torch.multinomial(probs, 1)

                # Execute actions and track results
                for j, game_idx in enumerate(active_indices):
                    if piece_done[j]:
                        continue

                    tid = token_ids[j].item()
                    action_ids_lists[j].append(tid)

                    action_char = TOKEN_TO_CHAR[tid]
                    action_name = ACTION_TO_ENGINE[action_char]
                    step_result = envs[game_idx].step(action_name)

                    total_rewards[game_idx] += step_result['reward']
                    total_steps[game_idx] += 1

                    if action_name in ('left', 'right'):
                        total_rewards[game_idx] += LR_PENALTY

                    if envs[game_idx].current_piece_name != current_piece_names[j]:
                        piece_placed[j] = True
                        piece_done[j] = True
                        total_rewards[game_idx] += PIECE_PLACED_BONUS
                        lc = envs[game_idx].total_lines - lines_before[j]
                        if lc > 0:
                            total_rewards[game_idx] += lc * LINE_CLEAR_BONUS

                    elif envs[game_idx].done:
                        piece_done[j] = True

                if all(piece_done):
                    break

                attention_mask = torch.cat([
                    attention_mask,
                    torch.ones(batch_size, 1, dtype=torch.long, device=model.device)
                ], dim=1)

                out = model(
                    token_ids,
                    attention_mask=attention_mask,
                    past_key_values=past_kv,
                    use_cache=True,
                )
                past_kv = out.past_key_values

        # Force drop for pieces not placed in 20 tokens
        for j, game_idx in enumerate(active_indices):
            if not piece_placed[j] and not envs[game_idx].done:
                envs[game_idx].step('drop')
                total_rewards[game_idx] += NO_PLACE_PENALTY
                total_steps[game_idx] += 1
                lc = envs[game_idx].total_lines - lines_before[j]
                if lc > 0:
                    total_rewards[game_idx] += lc * LINE_CLEAR_BONUS

            if action_ids_lists[j]:
                pieces_data[game_idx].append({
                    'prompt_ids': prompt_ids_list[j].unsqueeze(0).cpu(),
                    'action_ids': torch.tensor(action_ids_lists[j], dtype=torch.long),
                })

    results = []
    for i in range(num_games):
        results.append({
            'reward': total_rewards[i],
            'pieces': pieces_data[i],
            'total_steps': total_steps[i],
            'total_lines': envs[i].total_lines,
            'pieces_placed': pieces_counts[i],
            'final_board': envs[i].board_to_text(),
        })
    return results


def play_one_game(model, tokenizer, seed, temperature=TEMPERATURE):
    """Convenience wrapper: play a single game."""
    return play_games_batched(model, tokenizer, seed, num_games=1, temperature=temperature)[0]


def train_one_iteration(model, optimizer, seed, temperature=TEMPERATURE):
    """
    One GRPO iteration with timing.
    """
    # Phase 1: Batched rollout
    t_rollout = _time.time()
    games = play_games_batched(model, tokenizer, seed, GAMES_PER_ITER, temperature)
    t_rollout = _time.time() - t_rollout

    rewards = torch.tensor([g['reward'] for g in games], dtype=torch.float32)
    mean_r = rewards.mean().item()
    std_r = rewards.std().item()

    # Phase 2: Advantages
    if std_r < 1e-8:
        return {'mean_reward': mean_r, 'std_reward': 0.0, 'loss': 0.0,
                'avg_steps': sum(g['total_steps'] for g in games) / GAMES_PER_ITER,
                'avg_lines': sum(g['total_lines'] for g in games) / GAMES_PER_ITER,
                'avg_pieces': sum(g['pieces_placed'] for g in games) / GAMES_PER_ITER,
                't_rollout': t_rollout, 't_update': 0.0}

    advantages = ((rewards - rewards.mean()) / (rewards.std() + 1e-8)).tolist()

    # Phase 3: Update
    t_update = _time.time()
    optimizer.zero_grad()
    total_pieces = max(1, sum(len(g['pieces']) for g in games))
    total_loss = 0.0

    for game_idx, game in enumerate(games):
        adv = advantages[game_idx]
        for piece in game['pieces']:
            prompt = piece['prompt_ids'].to(model.device)
            actions = piece['action_ids'].to(model.device)
            if len(actions) == 0:
                continue

            full_input = torch.cat([prompt.squeeze(0), actions]).unsqueeze(0)
            logits = model(full_input).logits

            P = prompt.shape[-1]
            action_logits = logits[0, P-1 : P-1+len(actions), :]

            masked = torch.full_like(action_logits, float('-inf'))
            masked[:, ACTION_TOKEN_IDS] = action_logits[:, ACTION_TOKEN_IDS]
            log_probs = F.log_softmax(masked / temperature, dim=-1)

            selected = log_probs.gather(1, actions.unsqueeze(1)).squeeze(1)
            piece_loss = -(selected.sum() * adv) / total_pieces

            piece_loss.backward()
            total_loss += piece_loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    t_update = _time.time() - t_update

    return {
        'mean_reward': mean_r,
        'std_reward': std_r,
        'loss': total_loss,
        'avg_steps': sum(g['total_steps'] for g in games) / GAMES_PER_ITER,
        'avg_lines': sum(g['total_lines'] for g in games) / GAMES_PER_ITER,
        'avg_pieces': sum(g['pieces_placed'] for g in games) / GAMES_PER_ITER,
        't_rollout': t_rollout,
        't_update': t_update,
    }

# Smoke test
print("Smoke test: playing 1 game...")
test_game = play_one_game(model, tokenizer, seed=0)
print(f"Reward: {test_game['reward']:.1f}, Steps: {test_game['total_steps']}, "
      f"Pieces: {test_game['pieces_placed']}, Lines: {test_game['total_lines']}")
print("Training functions ready.")


In [ ]:
# Cell 5: Demo — UNTRAINED model plays one game
print("=== UNTRAINED MODEL ===\n")

game = play_one_game(model, tokenizer, seed=42)

print(f"Pieces: {game['pieces_placed']}/{MAX_PIECES_PER_GAME}")
print(f"Total actions: {game['total_steps']}")
print(f"Lines cleared: {game['total_lines']}")
print(f"Game reward: {game['reward']:+.1f}")
print(f"\nFinal board:")
print(game['final_board'])

untrained_reward = game['reward']
print(f"\nUntrained reward: {untrained_reward:+.1f}")


In [ ]:
# Cell 6: Training loop — 100 iterations of per-piece GRPO
import time

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-6,
    weight_decay=0.01,
)

history = []

print("Starting per-piece GRPO training...")
print(f"Config: {GAMES_PER_ITER} games/iter, max {MAX_PIECES_PER_GAME} pieces/game, "
      f"max {MAX_ACTIONS_PER_PIECE} tokens/piece, {NUM_ITERATIONS} iterations\n")

for iteration in range(NUM_ITERATIONS):
    stats = train_one_iteration(model, optimizer, seed=iteration)
    history.append(stats)

    print(f"[Iter {iteration:3d}] "
          f"reward={stats['mean_reward']:+8.1f} "
          f"std={stats['std_reward']:6.1f} "
          f"loss={stats['loss']:7.3f} "
          f"steps={stats['avg_steps']:5.1f} "
          f"lines={stats['avg_lines']:4.1f} "
          f"pieces={stats['avg_pieces']:4.1f} "
          f"roll={stats['t_rollout']:.1f}s "
          f"upd={stats['t_update']:.1f}s")

print("\nTraining complete!")


In [ ]:
# Cell 7: Plot reward curve
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

iters = range(len(history))

axes[0].plot(iters, [h['mean_reward'] for h in history])
axes[0].set_title('Mean Reward per Iteration')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Reward')

axes[1].plot(iters, [h['avg_lines'] for h in history])
axes[1].set_title('Avg Lines Cleared')
axes[1].set_xlabel('Iteration')

axes[2].plot(iters, [h['loss'] for h in history])
axes[2].set_title('Policy Gradient Loss')
axes[2].set_xlabel('Iteration')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print("Reward curve saved to reward_curve.png")

In [ ]:
# Cell 8: Demo — TRAINED model plays 3 games
print("=== TRAINED MODEL ===\n")

trained_rewards = []
for seed in [42, 123, 7]:
    game = play_one_game(model, tokenizer, seed=seed)
    print(f"Seed {seed}: reward={game['reward']:+.1f}, "
          f"steps={game['total_steps']}, lines={game['total_lines']}, "
          f"pieces={game['pieces_placed']}")
    trained_rewards.append(game['reward'])

avg_trained = sum(trained_rewards) / len(trained_rewards)
print(f"\n{'='*50}")
print(f"UNTRAINED reward (seed=42): {untrained_reward:+.1f}")
print(f"TRAINED avg reward (3 games): {avg_trained:+.1f}")
print(f"Improvement: {avg_trained - untrained_reward:+.1f}")
print('='*50)

if avg_trained > untrained_reward:
    print("Training improved the model!")
else:
    print("Model needs more training or tuning.")

In [ ]:
# Cell 9: Push trained model to HF Hub
model.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
tokenizer.push_to_hub("VortexedSquirrel/tetris-agent-grpo")
print("Model pushed to https://huggingface.co/VortexedSquirrel/tetris-agent-grpo")